# Notebook 02 — Image Preprocessing Pipeline
**Crop Disease Detection System** · IILM University, Greater Noida · 2025–26

This notebook **demonstrates and visualizes** each stage of the preprocessing pipeline defined in `src/preprocess.py` and `src/feature_extraction.py`.

**Pipeline Stages:**
1. Load raw leaf image
2. Resize to 128×128
3. Normalize pixel values to [0.0, 1.0]
4. Green-channel masking (remove background)
5. Otsu's binarization segmentation
6. GLCM texture feature extraction
7. Colour feature extraction
8. Build complete feature matrix and save to CSV

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

sys.path.insert(0, '../src')
from preprocess import load_image, resize_image, normalize_image, green_channel_mask, segment_image
from feature_extraction import extract_glcm_features, extract_color_features, build_feature_vector

%matplotlib inline
plt.rcParams['figure.dpi'] = 110
print('Imports successful.')

## 1 — Visualise Preprocessing Pipeline on a Sample Image
Change `SAMPLE_IMAGE_PATH` to any image from your dataset.

In [ ]:
# ── Update this path to point to a real image from your dataset ──
SAMPLE_IMAGE_PATH = '../data/images/Leaf_Spot/sample_001.jpg'

if not os.path.exists(SAMPLE_IMAGE_PATH):
    print(f'[INFO] Sample image not found at {SAMPLE_IMAGE_PATH}')
    print('       Download the DiaMOS dataset and update the path above.')
    print('       The pipeline code is fully defined below for reference.')
else:
    # Run each pipeline step
    img_raw       = load_image(SAMPLE_IMAGE_PATH)
    img_resized   = resize_image(img_raw)
    img_norm      = normalize_image(img_resized)
    img_masked    = green_channel_mask(img_norm)
    img_segmented = segment_image(img_masked)

    # Plot all stages
    stages = [
        ('1. Original',      img_raw),
        ('2. Resized 128×128', img_resized),
        ('3. Normalized',    img_norm),
        ('4. Green Masked',  img_masked),
        ('5. Segmented',     img_segmented),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    for ax, (title, img) in zip(axes, stages):
        disp = np.clip(img, 0, 1) if img.dtype == np.float32 else img
        ax.imshow(disp.astype(np.float32) if img.dtype != np.uint8 else img)
        ax.set_title(title, fontsize=9, fontweight='bold')
        ax.axis('off')

    plt.suptitle('Preprocessing Pipeline — Stage by Stage', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('Pipeline visualisation complete.')

## 2 — Green-Channel Masking — Explained
The green channel of the RGB image is isolated. Pixels where green dominates are retained; soil and sky pixels are zeroed out.

In [ ]:
# Explain green channel masking with channel histograms
# (Run this cell after loading a real image above)

print('GREEN-CHANNEL MASKING EXPLAINED')
print('='*45)
print('Formula: pixel is kept IF (G > R + 0.15) OR (G > B + 0.15)')
print()
print('This removes:')
print('  - Brown soil pixels     (R > G)')
print('  - Blue sky pixels       (B > G)')
print('  - White background      (R≈G≈B)')
print()
print('This keeps:')
print('  - Green leaf tissue     (G dominant)')
print('  - Yellow disease spots  (R+G dominant, filtered less aggressively)')

## 3 — GLCM Feature Extraction — Explained

In [ ]:
print('GLCM TEXTURE FEATURES')
print('='*55)

glcm_formulas = [
    ('Contrast',    'Σ (i-j)² × P(i,j)',           'Intensity variation between adjacent pixels'),
    ('Energy',      'Σ P(i,j)²',                    'Uniformity / textural smoothness'),
    ('Homogeneity', 'Σ P(i,j) / (1 + |i-j|)',      'Closeness of distribution to diagonal'),
    ('Correlation', 'Σ[(i-μᵢ)(j-μⱼ)P(i,j)]/(σᵢσⱼ)', 'Linear dependency between pixel pairs'),
]

for name, formula, meaning in glcm_formulas:
    print(f'  {name:<14} Formula: {formula}')
    print(f'               Captures: {meaning}')
    print()

print('Computed across 4 angles (0°, 45°, 90°, 135°) → averaged for rotation invariance.')

In [ ]:
print('COLOUR FEATURES')
print('='*55)
print('6 statistics computed from non-background pixels only:')
print()
print('  red_mean    — Mean of R channel   (disease yellowing → higher R)')
print('  green_mean  — Mean of G channel   (healthy leaf → higher G)')
print('  blue_mean   — Mean of B channel   (fungal spots → higher B)')
print('  red_std     — Std of R channel    (heterogeneous disease → higher std)')
print('  green_std   — Std of G channel')
print('  blue_std    — Std of B channel')
print()
print('TOTAL: 4 GLCM + 6 Colour = 10 features per image')

## 4 — Build Full Feature Matrix
Run `src/feature_extraction.py` to generate `data/features.csv`.
The cell below shows what that script does internally.

In [ ]:
# This shows the logic of feature_extraction.py
# To actually run it on the full dataset:
#
#   python src/feature_extraction.py
#
# Or uncomment the lines below:

# from feature_extraction import build_dataset_features
# df = build_dataset_features(data_dir='../data/images', output_csv='../data/features.csv')
# print(df.head())

print('Feature extraction pipeline summary:')
print()
print('For each image:')
steps = [
    'load_image()          → Load RGB array from disk',
    'resize_image()        → Resize to 128×128 pixels',
    'normalize_image()     → Scale pixel values to [0.0, 1.0]',
    'green_channel_mask()  → Remove background pixels',
    'segment_image()       → Otsu binarization + morphological cleanup',
    'extract_glcm_features()  → 4 GLCM texture values',
    'extract_color_features() → 6 RGB channel statistics',
    'build_feature_vector()   → Merge into 10-D dict',
]
for i, step in enumerate(steps, 1):
    print(f'  Step {i}: {step}')
print()
print('Output: data/features.csv  (shape: 3505 × 11)')

## Summary

| Stage | Operation | Output |
|---|---|---|
| 1 | Load image | Raw RGB array |
| 2 | Resize | 128×128×3 array |
| 3 | Normalize | float32 [0.0–1.0] |
| 4 | Green-channel mask | Background removed |
| 5 | Otsu segmentation | Only leaf tissue retained |
| 6 | GLCM features | 4 values |
| 7 | Colour features | 6 values |
| 8 | Feature vector | 10-D array per image |

→ Proceed to **Notebook 03** for model training and evaluation.